In [33]:
import os
import json
from dotenv import load_dotenv
from langchain_community.retrievers import BM25Retriever
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_classic.retrievers import EnsembleRetriever

In [34]:
load_dotenv()

True

![Diagram](./images/12_hybrid_rag.png)

# What is Hybrid RAG?

So far, we've relied on semantic search (using vector embeddings) to find documents that are conceptually similar to our query. This is powerful, but sometimes you need the precision of a good old-fashioned keyword search (also known as lexical search). Hybrid RAG combines both methods to get the best of both worlds.

# Set up Retrievers

A hybrid RAG system uses multiple retrievers to fetch documents. Here we combine a **vector-based (semantic)** retriever with a **keyword-based (BM25)** retriever.

- The **vector retriever** reuses the embeddings we already indexed into the `shopeasy-basic-rag` namespace in Notebooks 1 & 2 — no re-indexing needed.
- The **BM25 retriever** runs entirely in memory, so we build its corpus directly from the source knowledge base (`shopeasy_knowledge_base.json`) — no need to round-trip through Pinecone.

![Diagram](./images/13_two_retrivers.png)

In [35]:
# connect to the existing Pinecone namespace populated in Notebooks 1 & 2

# define the embeddings model — must match how the vectors were indexed (512 dims),
# otherwise the query vector won't match the index dimension.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)  # 512-dim vectors

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index(os.environ["PINECONE_INDEX_NAME"])

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="shopeasy-basic-rag",
)

### Vector search retreiver

This retriever performs a semantic search. It finds documents that are conceptually similar to the query, even if they don't share the exact same keywords. We configure it to return the top 3 most similar documents.

In [36]:

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

### Keyword search retreiver

Next, we set up a `BM25Retriever`. BM25 is a popular algorithm for information retrieval that ranks documents based on the frequency of the query terms in each document, while also accounting for document length. This is a "sparse" retrieval method because it relies on matching keywords.

In [37]:
# BM25 is in-memory, so we build its corpus straight from the source knowledge base
# We load each entry as a Document (carrying its metadata) and chunk it with the SAME splitter settings as Notebook 1,
# so BM25 and the vector index search the same units.
with open("shopeasy_knowledge_base.json") as f:
    kb = json.load(f)

raw_docs = [
    Document(page_content=entry["content"], metadata=entry["metadata"])
    for entry in kb
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # chunk size (characters)
    chunk_overlap=100,   # chunk overlap (characters)
    add_start_index=True,
)
docs = text_splitter.split_documents(raw_docs)

print(f"{len(raw_docs)} documents -> {len(docs)} chunks for BM25")

51 documents -> 193 chunks for BM25


In [38]:
# Now, we can create the BM25Retriever from these documents and set it to return the top 3 results.

bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 3 


# Retreival & Generation

With our retrievers in place, we can now define the generation part of our RAG pipeline.

In [40]:
#configure the llm
llm = ChatOpenAI(model="gpt-4.1-mini")

#set the prompt template (same support-assistant persona as Notebooks 1 & 2)
template = """You are a customer-support assistant for ShopEasy, an e-commerce platform.
Use the following pieces of retrieved internal knowledge-base context to help resolve the customer's issue.
If the context doesn't contain the answer, say you don't have that information rather than guessing.
Be concise and practical: state the likely cause and the next step the agent should take.

Context:
{context}

Customer issue: {question}

Support guidance:"""

rag_prompt_template = PromptTemplate.from_template(template)

![Diagram](./images/14_semantic_keywordsearch.png)

## Fusing both retrievers: the EnsembleRetriever

Each retriever returns its own ranked list. To get one unified result we need to **fuse** them. LangChain's `EnsembleRetriever` does this for us: it queries every retriever and combines their rankings with **weighted Reciprocal Rank Fusion (RRF)** — a document that ranks highly across *both* lists is rewarded most. The `weights` argument lets us lean toward semantic or keyword search; we start with an equal `[0.5, 0.5]`.

> RRF is only one way to merge/rerank results — we'll list other options at the end of the notebook.

We'll run three real support queries through it, each isolating one behaviour:

1. **Example 1 — a bare identifier (`ORD-97654`):** keyword search wins.
2. **Example 2 — plain natural language (no identifiers, and the customer's words don't match the KB's words):** semantic search wins.
3. **Example 3 — a mixed query (`ORD-91045` + "stuck refund"):** fusion wins — each retriever surfaces a document the other missed.

![Diagram](./images/15_rrf.png)

In [41]:
# The EnsembleRetriever runs both retrievers and fuses their ranked lists with
# weighted Reciprocal Rank Fusion (RRF) internally — so we no longer hand-roll RRF.
# `weights` lets us favour one signal over the other; here we weight them equally.
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5],  # equal weight to semantic and keyword search
)


# Helper: show each retriever's individual view, then the fused (ensemble) result,
# and return the fused docs so we can generate an answer from them.
def show_hybrid(query):
    vector_docs = vector_retriever.invoke(query)
    bm25_docs = bm25_retriever.invoke(query)
    fused_docs = ensemble_retriever.invoke(query)

    print(f"QUERY: {query}\n")
    print("VECTOR (semantic):")
    for doc in vector_docs:
        m = doc.metadata
        print(f"   [{m['doc_type']}/{m['product_area']}] {m['title']}")
    print("\nBM25 (keyword):")
    for doc in bm25_docs:
        m = doc.metadata
        print(f"   [{m['doc_type']}/{m['product_area']}] {m['title']}")
    print("\nENSEMBLE (fused):")
    for doc in fused_docs:
        m = doc.metadata
        print(f"   [{m['doc_type']}/{m['product_area']}] {m['title']}")
    return fused_docs

![Diagram](./images/16_query_behavior.png)

### Example 1 — an exact identifier (keyword search wins)

Real tickets are full of identifiers — order IDs, tracking numbers, SKUs, bug codes. When an agent pastes a bare order number like `ORD-97654` into search, the embedding model splits it into common sub-pieces (`ORD`, the digits) that **every** order shares — so semantic search drifts to *other* order tickets and never locks onto the right one. BM25 treats `ORD-97654` as a single exact token and matches the one ticket that contains it. The ensemble lets keyword search rescue the result.

In [42]:
# Example 1 — a bare exact identifier (an order number), exactly what an agent pastes from a ticket.
# The embedding model breaks "ORD-97654" into sub-pieces shared by every order id, so semantic
# search drifts to unrelated order tickets. Keyword (BM25) search matches the exact token and wins.
user_question = "ORD-97654"

In [43]:
# Run the ensemble for our first ticket (the helper also prints each retriever's view).
fused_docs = show_hybrid(user_question)

QUERY: ORD-97654

VECTOR (semantic):
   [bug_report/orders] Inventory sync delay causes overselling during flash sales
   [past_ticket/orders] Wrong color variant of SKU-CL-7834-RD received
   [runbook/orders] Resolving Inventory Sync Errors (INV_SYNC_ERR_700)

BM25 (keyword):
   [past_ticket/payments] Customer charged for cancelled order ORD-97654
   [past_ticket/payments] Customer charged for cancelled order ORD-97654
   [faq/account] How do I contact ShopEasy customer support?

ENSEMBLE (fused):
   [bug_report/orders] Inventory sync delay causes overselling during flash sales
   [past_ticket/payments] Customer charged for cancelled order ORD-97654
   [past_ticket/orders] Wrong color variant of SKU-CL-7834-RD received
   [past_ticket/payments] Customer charged for cancelled order ORD-97654
   [runbook/orders] Resolving Inventory Sync Errors (INV_SYNC_ERR_700)
   [faq/account] How do I contact ShopEasy customer support?


### Example 2 — plain natural language (semantic search wins)

The flip side of Example 1. No identifiers here — and notice the customer's words don't even match the knowledge base's words: they say *courier* and *parcel*, the KB says *carrier* and *package*. Keyword search can only match surface tokens, so it latches onto incidental shared words and drags in unrelated promo and account tickets. Semantic search understands that "courier left it at the door but it isn't there" *means* a misdelivery case, and surfaces the right shipping runbook and ticket. The ensemble keeps those semantic hits on top.

In [45]:
# Example 2 — plain natural language, no special tokens, and deliberately different vocabulary
# than the KB ("courier"/"parcel" vs "carrier"/"package"). BM25 can't bridge that gap and gets
# noisy; semantic search carries the query, and the ensemble reflects that.
show_hybrid("a shopper says the courier left their parcel at the door but it isn't there");

QUERY: a shopper says the courier left their parcel at the door but it isn't there

VECTOR (semantic):
   [runbook/shipping] Investigating Shipping Delays (SHIP_DELAYED_301)
   [past_ticket/shipping] Order ORD-88921 shows delivered but never received
   [runbook/shipping] Investigating Shipping Delays (SHIP_DELAYED_301)

BM25 (keyword):
   [past_ticket/shipping] Order ORD-88921 shows delivered but never received
   [past_ticket/payments] Promo code SUMMER25OFF not applying at checkout
   [past_ticket/account] Account locked after password reset — ACC_LOCKED_503

ENSEMBLE (fused):
   [past_ticket/shipping] Order ORD-88921 shows delivered but never received
   [runbook/shipping] Investigating Shipping Delays (SHIP_DELAYED_301)
   [past_ticket/payments] Promo code SUMMER25OFF not applying at checkout
   [runbook/shipping] Investigating Shipping Delays (SHIP_DELAYED_301)
   [past_ticket/account] Account locked after password reset — ACC_LOCKED_503


### Example 3 — a mixed query (hybrid wins) 🎯

This is the payoff. A real question often mixes **natural language** with an **exact identifier** — here, the *concept* of a stuck refund plus the order number `ORD-91045`. Watch what each retriever contributes:

- **Semantic search** understands "stuck refund" and pulls the matching past ticket for that order.
- **Keyword search** locks onto `ORD-91045` and also surfaces two docs semantic search missed entirely: the **Return Policy** and the root-cause **bug report** (refund API calls to Stripe not retried on timeout).

Neither retriever alone gives the full picture. RRF fuses them so the agent gets the specific case *and* the policy *and* the underlying bug — a better-grounded answer than either could produce alone.

In [46]:
# Example 3 — a MIXED query: the concept of a "stuck refund" (natural language)
# PLUS an exact order id (ORD-91045). This is where hybrid shines:
#   - VECTOR understands "stuck refund" and finds the matching past ticket.
#   - BM25 locks onto ORD-91045 and also surfaces docs vector missed: the Return Policy
#     and the root-cause bug report (Stripe refund calls not retried on timeout).
#   - RRF fuses them so the agent gets the specific case + the policy + the underlying bug.
user_question = "how do I get a stuck refund released for order ORD-91045"
fused_docs = show_hybrid(user_question)

# Generate the support answer from the fused context.
docs_content = "\n\n".join(doc.page_content for doc in fused_docs)
prompt = rag_prompt_template.invoke({"question": user_question, "context": docs_content})
response = llm.invoke(prompt)
print("\n" + "=" * 80 + "\n")
print(response.content)

QUERY: how do I get a stuck refund released for order ORD-91045

VECTOR (semantic):
   [past_ticket/returns] Refund not received after 10 business days for ORD-91045
   [past_ticket/returns] Refund not received after 10 business days for ORD-91045
   [past_ticket/payments] Customer charged for cancelled order ORD-97654

BM25 (keyword):
   [product_doc/returns] Return Policy
   [past_ticket/returns] Refund not received after 10 business days for ORD-91045
   [bug_report/returns] Refund API calls to Stripe not retried on timeout

ENSEMBLE (fused):
   [past_ticket/returns] Refund not received after 10 business days for ORD-91045
   [product_doc/returns] Return Policy
   [past_ticket/returns] Refund not received after 10 business days for ORD-91045
   [past_ticket/payments] Customer charged for cancelled order ORD-97654
   [bug_report/returns] Refund API calls to Stripe not retried on timeout


The refund for order ORD-91045 is likely stuck in the system. The next step is for the support a

## Other rerankers & fusion strategies

We used the `EnsembleRetriever`'s built-in **weighted RRF** — it's simple, fast, and needs no extra model. But it's not the only way to merge or reorder candidates. Depending on your latency/quality budget:

- **Cross-encoder rerankers** (Cohere Rerank, BGE-reranker, `sentence-transformers` cross-encoders): re-score each candidate by running the *query + document together* through a model — much more accurate than RRF, at the cost of a model call per candidate. In LangChain: `ContextualCompressionRetriever` wrapping a `CrossEncoderReranker` or `CohereRerank`.
- **LLM-based rerankers** (e.g. `LLMListwiseRerank`): ask an LLM to reorder the candidates. Flexible but slower and pricier.
- **Maximal Marginal Relevance (MMR)**: reranks for *diversity* to avoid near-duplicate chunks (`search_type="mmr"` on the vector retriever).
- **Native hybrid search**: some vector stores (including Pinecone) can run dense + sparse search in a single call and return one fused score, skipping client-side fusion entirely.

A common production pattern: use cheap hybrid retrieval (like this ensemble) to pull a generous candidate set, then a cross-encoder reranker to precisely reorder the top-k before generation.